# Stock Return Predictor
## Notebook 6: Improvements — Expanded S&P 500 Universe

**Project:** Predicting whether a stock will outperform the S&P 500 over the next month  
**Author:** Your Name  
**Date:** April 2026  

---
### Objective
Expand the baseline model from 20 hardcoded stocks to the full S&P 500 universe,
extending the history back to 2010. This increases training observations from ~1,500
to 85,000+, significantly improving model reliability.

### Changes from Baseline
- Universe: 20 stocks → ~500 S&P 500 constituents
- History: 2018–2024 → 2010–2024
- Top picks in backtest: 5 → 10

### Contents
1. Imports & Setup
2. Pull S&P 500 Tickers
3. Download Price Data
4. Feature Engineering
5. Modeling
6. Backtesting
7. Compare Results to Baseline

## 1. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import os

from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from xgboost import XGBRegressor

DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)

## 2. Pull S&P 500 Tickers
Dynamically pull current S&P 500 constituents from Wikipedia.

In [ ]:
import ssl
import urllib.request
from io import StringIO

# Fix SSL certificate issue on Mac
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

try:
    with urllib.request.urlopen(req, context=ssl_context) as response:
        html = response.read().decode("utf-8")  # decode bytes to string
    tables = pd.read_html(StringIO(html))        # wrap in StringIO
    TICKERS = tables[0]["Symbol"].tolist()
    TICKERS = [t.replace(".", "-") for t in TICKERS]
    print(f"Total tickers pulled: {len(TICKERS)}")
except Exception as e:
    print(f"Failed: {e}")

## 3. Download Price Data
Pull daily adjusted closing prices from 2010 onwards.


In [ ]:
START_DATE = "2010-01-01"
END_DATE   = "2024-12-31"

raw_df = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)
print(f"Raw data shape: {raw_df.shape}")

In [ ]:
# Keep only closing prices
prices_sp500 = raw_df["Close"]

# Drop tickers with more than 20% missing data
threshold = 0.2 * len(prices_sp500)
prices_sp500 = prices_sp500.dropna(axis=1, thresh=len(prices_sp500) - threshold)

print(f"Prices shape after cleaning: {prices_sp500.shape}")
print(f"Tickers remaining: {prices_sp500.shape[1]}")

prices_sp500.to_csv(f"{DATA_DIR}/prices_sp500.csv")
print("Saved prices_sp500.csv")

## 4. Feature Engineering
Same 6 features as baseline — momentum, moving average ratios, volatility.
Target variable: did this stock beat SPY next month?

In [ ]:
# Resample to monthly
monthly_prices = prices_sp500.resample("ME").last()
monthly_returns = monthly_prices.pct_change()

print(f"Monthly prices shape: {monthly_prices.shape}")

In [ ]:
# Pull SPY as benchmark
spy_raw     = yf.download("SPY", start=START_DATE, end=END_DATE, auto_adjust=True)
spy_monthly = spy_raw["Close"].resample("ME").last()
spy_returns = spy_monthly.pct_change().dropna()

print(f"SPY monthly returns shape: {spy_returns.shape}")

In [ ]:
FEATURE_COLS = [
    "ret_1m",
    "ret_3m",
    "ret_6m",
    "ma_ratio_20",
    "ma_ratio_50",
    "volatility",
    "zscore_20"
]

TICKERS_CLEAN = prices_sp500.columns.tolist()

def compute_features_and_target(prices_daily, monthly_prices, monthly_returns, spy_returns):
    features_list = []

    for date in monthly_prices.index[6:]:
        future_dates = spy_returns.index[spy_returns.index > date]
        if len(future_dates) == 0:
            continue

        next_month = future_dates[0]
        spy_next = float(spy_returns.loc[next_month].item()) if hasattr(spy_returns.loc[next_month], 'item') else float(spy_returns.loc[next_month])

        for ticker in monthly_prices.columns:
            hist = prices_daily[ticker].dropna().loc[:date]
            if len(hist) < 126:
                continue

            price_today = hist.iloc[-1]

            # --- Momentum ---
            ret_1m  = (price_today / hist.iloc[-21]  - 1) if len(hist) >= 21  else np.nan
            ret_3m  = (price_today / hist.iloc[-63]  - 1) if len(hist) >= 63  else np.nan
            ret_6m  = (price_today / hist.iloc[-126] - 1) if len(hist) >= 126 else np.nan

            # --- Moving averages ---
            ma_20 = hist.iloc[-20:].mean()
            ma_50 = hist.iloc[-50:].mean()
            ma_ratio_20 = price_today / ma_20 - 1
            ma_ratio_50 = price_today / ma_50 - 1

            # --- Volatility ---
            daily_rets = hist.pct_change().iloc[-20:]
            volatility = daily_rets.std() * np.sqrt(252)

            # --- NEW: Mean reversion (z-score) ---
            zscore_20 = (price_today - ma_20) / hist.iloc[-20:].std()

            # --- Target: excess return vs SPY ---
            stock_next = monthly_returns[ticker].get(next_month, np.nan)
            if hasattr(stock_next, 'item'):
                stock_next = stock_next.item()

            excess_ret = stock_next - spy_next

            features_list.append({
                "date":        date,
                "ticker":      ticker,
                "ret_1m":      ret_1m,
                "ret_3m":      ret_3m,
                "ret_6m":      ret_6m,
                "ma_ratio_20": ma_ratio_20,
                "ma_ratio_50": ma_ratio_50,
                "volatility":  volatility,
                "zscore_20":   zscore_20,
                "target":      excess_ret
            })

    return pd.DataFrame(features_list).set_index("date")


print("Computing features — this will take several minutes...")
model_df = compute_features_and_target(prices_sp500, monthly_prices, monthly_returns, spy_returns)

print(f"Model dataframe shape: {model_df.shape}")
model_df.head()

In [ ]:
# Drop any NaNs
model_df = model_df.dropna()
model_df[FEATURE_COLS] = model_df.groupby("date")[FEATURE_COLS].rank(pct=True)
print(f"After dropping NaNs: {model_df.shape}")
print(f"Target mean: {model_df['target'].mean():.6f}")
print(f"Target std:  {model_df['target'].std():.6f}")
print(f"Average % beating SPY: {model_df['target'].mean():.2%}")

model_df.to_csv(f"{DATA_DIR}/features_sp500.csv")
print("Saved features_sp500.csv")

## 5. Modeling
Same three models as baseline. With 85,000+ rows we expect meaningfully
better signal than the baseline's near-random results.

In [ ]:
## 5. Modeling: Ridge vs. XGBoost
# ---------------------------------------------------------
# 1. Ridge Baseline (Linear)
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)
ridge_preds = ridge_model.predict(X_test_scaled)

# 2. XGBoost Challenger (Non-Linear)
# We use conservative parameters to handle the noise of stock data
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_scaled, y_train)
xgb_preds = xgb_model.predict(X_test_scaled)

# Store both in the test_df for comparison
test_df = test_df.copy()
test_df["pred_ridge"] = ridge_preds
test_df["pred_xgb"] = xgb_preds

# Print the "Correlation Fight"
ridge_corr = np.corrcoef(y_test, ridge_preds)[0, 1]
xgb_corr = np.corrcoef(y_test, xgb_preds)[0, 1]
print(f"Ridge Correlation:   {ridge_corr:.4f}")
print(f"XGBoost Correlation: {xgb_corr:.4f}")

In [ ]:
## 5.1 Decile Analysis Comparison
# ---------------------------------------------------------
# Rank and bucket both models
test_df["decile_ridge"] = test_df.groupby(level=0)["pred_ridge"].transform(lambda x: pd.qcut(x, 10, labels=False))
test_df["decile_xgb"] = test_df.groupby(level=0)["pred_xgb"].transform(lambda x: pd.qcut(x, 10, labels=False))

# Calculate mean excess returns
ridge_res = test_df.groupby("decile_ridge")["target"].mean()
xgb_res = test_df.groupby("decile_xgb")["target"].mean()

# Plot
comparison_df = pd.DataFrame({"Ridge": ridge_res, "XGBoost": xgb_res})
comparison_df.plot(kind="bar", figsize=(10, 5), color=["lightgrey", "steelblue"])
plt.title("Which model picks the best winners? (Decile 9)")
plt.ylabel("Avg Monthly Excess Return")
plt.show()

## 6. Backtesting
Rolling window backtest with top 10 picks per month instead of 5.

In [ ]:
## 7. Comparison: Growth of $1
# ---------------------------------------------------------
plt.figure(figsize=(14, 6))

(1 + ridge_series).cumprod().plot(label="Ridge Strategy", color="lightgrey", linewidth=2)
(1 + xgb_series).cumprod().plot(label="XGBoost Strategy", color="steelblue", linewidth=2)
(1 + spy_aligned).cumprod().plot(label="SPY Benchmark", color="orange", linestyle="--")

plt.title("Backtest Comparison: Ridge vs. XGBoost vs. SPY")
plt.ylabel("Cumulative Returns")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
portfolio_returns = {}

for date, tickers in strategy_returns.items():
    if date not in monthly_returns.index:
        continue

    top, bottom = tickers

    top_rets = [monthly_returns.loc[date, t] for t in top if t in monthly_returns.columns]
    bot_rets = [monthly_returns.loc[date, t] for t in bottom if t in monthly_returns.columns]

    if top_rets and bot_rets:
        portfolio_returns[date] = np.mean(top_rets) - np.mean(bot_rets)

portfolio_series = pd.Series(portfolio_returns).sort_index()
spy_aligned      = spy_returns.reindex(portfolio_series.index).dropna()
portfolio_series = portfolio_series.reindex(spy_aligned.index)

print(f"Backtest months: {len(portfolio_series)}")
print(f"Portfolio mean monthly return: {portfolio_series.mean():.4f}")
print(f"SPY mean monthly return:       {spy_aligned.values.flatten().mean():.4f}")

## 7. Compare Results to Baseline

In [ ]:
# Cumulative returns
portfolio_cumulative = (1 + portfolio_series).cumprod()
spy_cumulative       = (1 + spy_aligned).cumprod()

portfolio_total  = float(portfolio_cumulative.iloc[-1]) - 1
spy_total        = float(spy_cumulative.iloc[-1].item()) - 1
n_months         = len(portfolio_series)
portfolio_annual = (1 + portfolio_total) ** (12 / n_months) - 1
spy_annual       = (1 + spy_total)       ** (12 / n_months) - 1
portfolio_sharpe = (portfolio_series.mean() / portfolio_series.std()) * np.sqrt(12)
spy_sharpe       = (spy_aligned.values.flatten().mean() / spy_aligned.values.flatten().std()) * np.sqrt(12)

def max_drawdown(cum):
    return ((cum - cum.cummax()) / cum.cummax()).min()

portfolio_dd = float(max_drawdown(portfolio_cumulative))
spy_dd       = float(max_drawdown(spy_cumulative).item())

comparison = pd.DataFrame({
    "Metric": ["Total Return", "Annualized Return", "Sharpe Ratio", "Max Drawdown"],
    "Baseline Strategy": ["238.45%", "32.49%", "1.275", "-17.73%"],
    "Improved Strategy": [
        f"{portfolio_total:.2%}",
        f"{portfolio_annual:.2%}",
        f"{portfolio_sharpe:.3f}",
        f"{portfolio_dd:.2%}"
    ],
    "SPY": [
        f"{spy_total:.2%}",
        f"{spy_annual:.2%}",
        f"{spy_sharpe:.3f}",
        f"{spy_dd:.2%}"
    ]
})

print(comparison.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(portfolio_cumulative.index, portfolio_cumulative.values,
        label="Improved Strategy", color="steelblue", linewidth=2)
ax.plot(spy_cumulative.index, spy_cumulative.values.flatten(),
        label="SPY", color="orange", linewidth=2, linestyle="--")
ax.set_title("Cumulative Returns: Improved Strategy vs SPY")
ax.set_ylabel("Growth of $1")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()